In [5]:
import pandas as pd
import numpy as np

customers = pd.read_csv("../data/raw/customers.csv")
transactions = pd.read_csv("../data/raw/transactions.csv")
recovery = pd.read_csv("../data/raw/recovery_outcomes.csv")

print("Customers:", customers.shape)
print("Transactions:", transactions.shape)
print("Recovery outcomes:", recovery.shape)

Customers: (2500, 13)
Transactions: (10000, 20)
Recovery outcomes: (11823, 10)


In [6]:
print("===== CUSTOMERS =====")
print(customers.info())

print("\n===== TRANSACTIONS =====")
print(transactions.info())

print("\n===== RECOVERY OUTCOMES =====")
print(recovery.info())

===== CUSTOMERS =====
<class 'pandas.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 13 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   customer_id                 2500 non-null   str    
 1   account_age_days            2500 non-null   int64  
 2   total_transactions          2500 non-null   int64  
 3   successful_transactions     2500 non-null   int64  
 4   failed_transactions         2500 non-null   int64  
 5   historical_success_rate     2500 non-null   float64
 6   average_transaction_value   2500 non-null   float64
 7   median_transaction_value    2500 non-null   float64
 8   total_spend                 2500 non-null   float64
 9   days_since_last_success     2500 non-null   int64  
 10  previous_recovery_attempts  2500 non-null   int64  
 11  previous_recoveries         2500 non-null   int64  
 12  customer_segment            2500 non-null   str    
dtypes: float64(4), int64(7

In [7]:
print("===== MISSING VALUES =====")

print("\nCUSTOMERS")
print(customers.isnull().sum())

print("\nTRANSACTIONS")
print(transactions.isnull().sum())

print("\nRECOVERY OUTCOMES")
print(recovery.isnull().sum())

===== MISSING VALUES =====

CUSTOMERS
customer_id                   0
account_age_days              0
total_transactions            0
successful_transactions       0
failed_transactions           0
historical_success_rate       0
average_transaction_value     0
median_transaction_value      0
total_spend                   0
days_since_last_success       0
previous_recovery_attempts    0
previous_recoveries           0
customer_segment              0
dtype: int64

TRANSACTIONS
transaction_id              0
customer_id                 0
order_id                    0
amount                      0
currency                    0
payment_method              0
timestamp                   0
status                      0
failure_reason           7049
failure_source           7049
failure_step             7049
attempt_number              0
checkout_started            0
checkout_duration_sec       0
is_subscription             0
subscription_status      9507
device_type                 0
is_new_de

In [11]:
print("===== RESTORING TRANSACTION-LEVEL SPLIT =====")

from sklearn.model_selection import train_test_split

# Get unique transaction IDs
unique_transaction_ids = ml_data["transaction_id"].unique()

# Recreate the exact original split
train_transaction_ids, test_transaction_ids = train_test_split(
    unique_transaction_ids,
    test_size=0.20,
    random_state=42
)

print("Unique transactions:", len(unique_transaction_ids))
print("Training transactions:", len(train_transaction_ids))
print("Testing transactions:", len(test_transaction_ids))

# Verify expected split
assert len(train_transaction_ids) == 3152
assert len(test_transaction_ids) == 789

print("\n===== SPLIT RESTORED SUCCESSFULLY =====")

===== RESTORING TRANSACTION-LEVEL SPLIT =====
Unique transactions: 3941
Training transactions: 3152
Testing transactions: 789

===== SPLIT RESTORED SUCCESSFULLY =====


In [12]:
print("===== REBUILDING ACTION-SPECIFIC MODELS =====")

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

# Feature groups used by the action-specific models
numeric_features_action = [
    "amount",
    "attempt_number",
    "checkout_duration_sec",
    "hour_of_day",
    "day_of_week",
    "account_age_days",
    "total_transactions",
    "successful_transactions",
    "failed_transactions",
    "historical_success_rate",
    "average_transaction_value",
    "median_transaction_value",
    "total_spend",
    "days_since_last_success",
    "previous_recovery_attempts",
    "previous_recoveries",
    "checkout_started",
    "is_subscription",
    "is_new_device"
]

categorical_features_action = [
    "currency",
    "payment_method",
    "status",
    "failure_reason",
    "failure_source",
    "failure_step",
    "subscription_status",
    "device_type",
    "customer_segment"
]

action_models = {}

for action in ["retry", "reminder", "escalation"]:

    print(f"\nTraining model for: {action}")

    action_data = ml_data[
        (ml_data["action"] == action) &
        (ml_data["transaction_id"].isin(train_transaction_ids))
    ].copy()

    X_action = action_data[
        numeric_features_action +
        categorical_features_action
    ].copy()

    y_action = action_data["action_success"].copy()

    # Convert boolean features to integers
    for column in [
        "checkout_started",
        "is_subscription",
        "is_new_device"
    ]:
        X_action[column] = X_action[column].astype(int)

    numeric_transformer_action = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_transformer_action = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ))
    ])

    preprocessor_action = ColumnTransformer([
        (
            "num",
            numeric_transformer_action,
            numeric_features_action
        ),
        (
            "cat",
            categorical_transformer_action,
            categorical_features_action
        )
    ])

    model = Pipeline([
        ("preprocessor", preprocessor_action),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ])

    model.fit(X_action, y_action)

    action_models[action] = model

    print("Training rows:", len(X_action))
    print("Model trained successfully.")

print("\n===== ACTION MODELS REBUILT =====")
print(list(action_models.keys()))

===== REBUILDING ACTION-SPECIFIC MODELS =====

Training model for: retry
Training rows: 3152
Model trained successfully.

Training model for: reminder
Training rows: 3152
Model trained successfully.

Training model for: escalation
Training rows: 3152
Model trained successfully.

===== ACTION MODELS REBUILT =====
['retry', 'reminder', 'escalation']


In [13]:
print("===== GENERATING ACTION PROBABILITIES =====")

# Build the test transaction feature set
test_data = ml_data[
    ml_data["transaction_id"].isin(test_transaction_ids)
].copy()

# Keep one transaction-level feature row per transaction
test_features = (
    test_data
    .drop_duplicates("transaction_id")
    .set_index("transaction_id")
)

# Features used by the action-specific models
model_features = (
    numeric_features_action +
    categorical_features_action
)

X_test_action = test_features[model_features].copy()

# Convert boolean features to integers
for column in [
    "checkout_started",
    "is_subscription",
    "is_new_device"
]:
    X_test_action[column] = X_test_action[column].astype(int)

# Generate probabilities
action_probabilities = pd.DataFrame(
    index=test_features.index
)

for action in ["retry", "reminder", "escalation"]:

    print(f"Predicting: {action}")

    model = action_models[action]

    action_probabilities[f"p_{action}"] = (
        model.predict_proba(X_test_action)[:, 1]
    )

# Add transaction amount
action_probabilities["amount"] = test_features["amount"]

# Convert transaction ID from index to column
action_probabilities = action_probabilities.reset_index()

print("\n===== ACTION PROBABILITIES READY =====")

print("Shape:", action_probabilities.shape)

print("\nColumns:")
print(action_probabilities.columns.tolist())

print("\n===== PROBABILITY SUMMARY =====")
print(
    action_probabilities[
        ["p_retry", "p_reminder", "p_escalation"]
    ]
    .describe()
    .round(4)
)

print("\n===== SAMPLE PREDICTIONS =====")
print(
    action_probabilities.head(10).round(4)
)

print("\n===== GENERATION COMPLETE =====")

===== GENERATING ACTION PROBABILITIES =====
Predicting: retry
Predicting: reminder
Predicting: escalation

===== ACTION PROBABILITIES READY =====
Shape: (789, 5)

Columns:
['transaction_id', 'p_retry', 'p_reminder', 'p_escalation', 'amount']

===== PROBABILITY SUMMARY =====
        p_retry  p_reminder  p_escalation
count  789.0000    789.0000      789.0000
mean     0.6574      0.4863        0.5327
std      0.1403      0.1203        0.1126
min      0.2093      0.1886        0.2039
25%      0.5747      0.3937        0.4577
50%      0.6841      0.4805        0.5411
75%      0.7572      0.5853        0.6100
max      0.9430      0.7521        0.8187

===== SAMPLE PREDICTIONS =====
  transaction_id  p_retry  p_reminder  p_escalation     amount
0     TXN1000020   0.6510      0.4601        0.6526  116967.92
1     TXN1000025   0.7635      0.7290        0.7079    2884.05
2     TXN1000035   0.8358      0.5513        0.6768    1603.29
3     TXN1000040   0.3848      0.3449        0.3112   30100.03


In [14]:
print("===== REVENUESHIELD EXPECTED-VALUE DECISION =====")

# Intervention costs
costs = {
    "retry": 2,
    "reminder": 1,
    "escalation": 15
}

# Calculate expected net recovery for each candidate action
action_probabilities["expected_retry"] = (
    action_probabilities["p_retry"]
    * action_probabilities["amount"]
    - costs["retry"]
)

action_probabilities["expected_reminder"] = (
    action_probabilities["p_reminder"]
    * action_probabilities["amount"]
    - costs["reminder"]
)

action_probabilities["expected_escalation"] = (
    action_probabilities["p_escalation"]
    * action_probabilities["amount"]
    - costs["escalation"]
)

expected_columns = [
    "expected_retry",
    "expected_reminder",
    "expected_escalation"
]

# Select the action with the highest expected net recovery
action_probabilities["selected_action"] = (
    action_probabilities[expected_columns]
    .idxmax(axis=1)
    .str.replace("expected_", "", regex=False)
)

action_probabilities["expected_net_recovery"] = (
    action_probabilities[expected_columns].max(axis=1)
)

print("\n===== SELECTED ACTION DISTRIBUTION =====")

selected_distribution = (
    action_probabilities["selected_action"]
    .value_counts()
    .reindex(["retry", "reminder", "escalation"])
    .fillna(0)
    .astype(int)
)

print(selected_distribution)

print("\n===== SELECTED ACTION PERCENTAGE =====")

print(
    selected_distribution
    .div(len(action_probabilities))
    .mul(100)
    .round(2)
)

print("\n===== EXPECTED NET RECOVERY =====")

print(
    action_probabilities["expected_net_recovery"]
    .describe()
    .round(2)
)

print("\n===== EXPECTED NET RECOVERY BY SELECTED ACTION =====")

print(
    action_probabilities
    .groupby("selected_action")["expected_net_recovery"]
    .agg(["count", "mean", "median", "sum"])
    .reindex(["retry", "reminder", "escalation"])
    .round(2)
)

print("\n===== SAMPLE REVENUESHIELD DECISIONS =====")

print(
    action_probabilities[
        [
            "transaction_id",
            "amount",
            "p_retry",
            "p_reminder",
            "p_escalation",
            "expected_retry",
            "expected_reminder",
            "expected_escalation",
            "selected_action",
            "expected_net_recovery"
        ]
    ]
    .head(10)
    .round(2)
)

print("\n===== DECISION ENGINE COMPLETE =====")

===== REVENUESHIELD EXPECTED-VALUE DECISION =====

===== SELECTED ACTION DISTRIBUTION =====
selected_action
retry         729
reminder       10
escalation     50
Name: count, dtype: int64

===== SELECTED ACTION PERCENTAGE =====
selected_action
retry         92.40
reminder       1.27
escalation     6.34
Name: count, dtype: float64

===== EXPECTED NET RECOVERY =====
count      789.00
mean      4810.31
std       6256.45
min        167.04
25%       1388.67
50%       2793.23
75%       5454.98
max      76318.39
Name: expected_net_recovery, dtype: float64

===== EXPECTED NET RECOVERY BY SELECTED ACTION =====
                 count     mean   median         sum
selected_action                                     
retry              729  4730.59  2810.53  3448603.12
reminder            10  2240.98  1459.17    22409.82
escalation          50  6486.50  2584.59   324325.09

===== SAMPLE REVENUESHIELD DECISIONS =====
  transaction_id     amount  p_retry  p_reminder  p_escalation  \
0     TXN1000020

In [15]:
print("===== REVENUESHIELD ACTUAL ECONOMIC PERFORMANCE =====")

# Actual recovery outcomes for the test transactions
test_recovery = recovery[
    recovery["transaction_id"].isin(test_transaction_ids)
].copy()

# Get the action selected by RevenueShield
selected_actions = action_probabilities[
    ["transaction_id", "selected_action"]
].copy()

# Merge selected action with actual recovery outcomes
evaluation = selected_actions.merge(
    test_recovery[
        [
            "transaction_id",
            "action",
            "action_success",
            "amount_recovered",
            "intervention_cost"
        ]
    ],
    left_on=["transaction_id", "selected_action"],
    right_on=["transaction_id", "action"],
    how="left",
    validate="one_to_one"
)

# Actual net recovery
evaluation["actual_net_recovery"] = (
    evaluation["amount_recovered"]
    - evaluation["intervention_cost"]
)

print("\n===== DATA VALIDATION =====")
print("Transactions evaluated:", len(evaluation))
print("Missing recovery outcomes:",
      evaluation["action_success"].isna().sum())

print("\n===== SELECTED ACTION DISTRIBUTION =====")
print(
    evaluation["selected_action"]
    .value_counts()
    .reindex(["retry", "reminder", "escalation"])
    .fillna(0)
    .astype(int)
)

print("\n===== ACTUAL PERFORMANCE =====")

total_recovery = evaluation["actual_net_recovery"].sum()
average_recovery = evaluation["actual_net_recovery"].mean()

print(
    f"RevenueShield total net recovery: ₹{total_recovery:,.2f}"
)

print(
    f"RevenueShield average net recovery: ₹{average_recovery:,.2f}"
)

print(
    f"RevenueShield success rate: "
    f"{evaluation['action_success'].mean() * 100:.2f}%"
)

print("\n===== PERFORMANCE BY SELECTED ACTION =====")

performance_by_action = (
    evaluation
    .groupby("selected_action")
    .agg(
        transactions=("transaction_id", "count"),
        successful_recoveries=("action_success", "sum"),
        success_rate=("action_success", "mean"),
        total_net_recovery=("actual_net_recovery", "sum"),
        average_net_recovery=("actual_net_recovery", "mean")
    )
    .reindex(["retry", "reminder", "escalation"])
)

performance_by_action["success_rate"] *= 100

print(
    performance_by_action.round(2)
)

print("\n===== REVENUESHIELD ACTUAL ECONOMIC PERFORMANCE COMPLETE =====")


===== REVENUESHIELD ACTUAL ECONOMIC PERFORMANCE =====

===== DATA VALIDATION =====
Transactions evaluated: 789
Missing recovery outcomes: 0

===== SELECTED ACTION DISTRIBUTION =====
selected_action
retry         729
reminder       10
escalation     50
Name: count, dtype: int64

===== ACTUAL PERFORMANCE =====
RevenueShield total net recovery: ₹3,454,938.99
RevenueShield average net recovery: ₹4,378.88
RevenueShield success rate: 63.62%

===== PERFORMANCE BY SELECTED ACTION =====
                 transactions  successful_recoveries  success_rate  \
selected_action                                                      
retry                     729                    478         65.57   
reminder                   10                      5         50.00   
escalation                 50                     19         38.00   

                 total_net_recovery  average_net_recovery  
selected_action                                            
retry                    3267219.31           

In [16]:
print("===== DECISION MARGIN ANALYSIS =====")

# Sort expected values for every transaction
expected_values = action_probabilities[
    [
        "expected_retry",
        "expected_reminder",
        "expected_escalation"
    ]
].copy()

sorted_values = expected_values.apply(
    lambda row: sorted(row, reverse=True),
    axis=1,
    result_type="expand"
)

sorted_values.columns = [
    "best_expected_value",
    "second_best_expected_value",
    "third_best_expected_value"
]

action_probabilities["decision_margin"] = (
    sorted_values["best_expected_value"]
    - sorted_values["second_best_expected_value"]
)

action_probabilities["relative_margin"] = (
    action_probabilities["decision_margin"]
    / action_probabilities["amount"]
)

print("\n===== DECISION MARGIN SUMMARY =====")

print(
    action_probabilities["decision_margin"]
    .describe()
    .round(2)
)

print("\n===== RELATIVE MARGIN SUMMARY =====")

print(
    action_probabilities["relative_margin"]
    .describe()
    .round(4)
)

print("\n===== MARGIN PERCENTILES =====")

print(
    action_probabilities["decision_margin"]
    .quantile(
        [0.10, 0.25, 0.50, 0.75, 0.90]
    )
    .round(2)
)

print("\n===== DECISION MARGIN BY SELECTED ACTION =====")

print(
    action_probabilities
    .groupby("selected_action")["decision_margin"]
    .agg(["count", "mean", "median", "min", "max"])
    .reindex(["retry", "reminder", "escalation"])
    .round(2)
)

print("\n===== LOW-CONFIDENCE DECISIONS =====")

print(
    action_probabilities[
        [
            "transaction_id",
            "amount",
            "p_retry",
            "p_reminder",
            "p_escalation",
            "expected_retry",
            "expected_reminder",
            "expected_escalation",
            "selected_action",
            "decision_margin"
        ]
    ]
    .sort_values("decision_margin")
    .head(15)
    .round(2)
)

print("\n===== DECISION MARGIN ANALYSIS COMPLETE =====")

===== DECISION MARGIN ANALYSIS =====

===== DECISION MARGIN SUMMARY =====
count      789.00
mean       794.98
std       1185.97
min          0.59
25%        189.57
50%        445.42
75%        950.73
max      16460.17
Name: decision_margin, dtype: float64

===== RELATIVE MARGIN SUMMARY =====
count    789.0000
mean       0.1164
std        0.0701
min        0.0003
25%        0.0645
50%        0.1078
75%        0.1633
max        0.3853
Name: relative_margin, dtype: float64

===== MARGIN PERCENTILES =====
0.10      78.11
0.25     189.57
0.50     445.42
0.75     950.73
0.90    1803.92
Name: decision_margin, dtype: float64

===== DECISION MARGIN BY SELECTED ACTION =====
                 count    mean  median   min       max
selected_action                                       
retry              729  833.29  474.50  0.59  16460.17
reminder            10   89.23   55.19  2.14    304.25
escalation          50  377.50  172.37  2.03   1984.11

===== LOW-CONFIDENCE DECISIONS =====
    transactio

In [17]:
print("===== CONSERVATIVE POLICY THRESHOLD ANALYSIS =====")

# Merge actual outcomes into our decision table
policy_test = action_probabilities[
    [
        "transaction_id",
        "amount",
        "selected_action",
        "decision_margin"
    ]
].copy()

policy_test = policy_test.merge(
    recovery[
        [
            "transaction_id",
            "action",
            "action_success",
            "amount_recovered",
            "intervention_cost"
        ]
    ],
    left_on=["transaction_id", "selected_action"],
    right_on=["transaction_id", "action"],
    how="left",
    validate="one_to_one"
)

# Actual net recovery under the original RevenueShield decision
policy_test["original_net_recovery"] = (
    policy_test["amount_recovered"]
    - policy_test["intervention_cost"]
)

# Get actual retry outcomes for fallback
retry_outcomes = recovery[
    recovery["action"] == "retry"
][
    [
        "transaction_id",
        "action_success",
        "amount_recovered",
        "intervention_cost"
    ]
].copy()

retry_outcomes = retry_outcomes.rename(
    columns={
        "action_success": "retry_success",
        "amount_recovered": "retry_amount_recovered",
        "intervention_cost": "retry_intervention_cost"
    }
)

policy_test = policy_test.merge(
    retry_outcomes,
    on="transaction_id",
    how="left",
    validate="one_to_one"
)

policy_test["retry_net_recovery"] = (
    policy_test["retry_amount_recovered"]
    - policy_test["retry_intervention_cost"]
)

# Always-Retry benchmark
always_retry_total = policy_test["retry_net_recovery"].sum()

print(f"\nAlways Retry baseline: ₹{always_retry_total:,.2f}")

# Thresholds to test
thresholds = [
    0,
    10,
    25,
    50,
    100,
    250,
    500,
    750,
    1000,
    1500,
    2000
]

results = []

for threshold in thresholds:

    # Start with original model decision
    policy_test["policy_action"] = (
        policy_test["selected_action"]
    )

    # Conservative fallback
    fallback_mask = (
        (policy_test["selected_action"] != "retry") &
        (policy_test["decision_margin"] < threshold)
    )

    policy_test.loc[
        fallback_mask,
        "policy_action"
    ] = "retry"

    # Select actual outcome corresponding to policy action
    policy_test["policy_net_recovery"] = (
        policy_test["original_net_recovery"]
    )

    policy_test.loc[
        fallback_mask,
        "policy_net_recovery"
    ] = policy_test.loc[
        fallback_mask,
        "retry_net_recovery"
    ]

    total = policy_test["policy_net_recovery"].sum()

    improvement = (
        (total - always_retry_total)
        / always_retry_total
        * 100
    )

    results.append({
        "threshold": threshold,
        "fallback_transactions": int(fallback_mask.sum()),
        "total_net_recovery": total,
        "improvement_vs_always_retry_pct": improvement
    })

threshold_results = pd.DataFrame(results)

print("\n===== THRESHOLD RESULTS =====")

print(
    threshold_results.round(2).to_string(index=False)
)

print("\n===== BEST THRESHOLD =====")

best_threshold = threshold_results.loc[
    threshold_results["total_net_recovery"].idxmax()
]

print(best_threshold.round(2))

print("\n===== CONSERVATIVE POLICY ANALYSIS COMPLETE =====")

===== CONSERVATIVE POLICY THRESHOLD ANALYSIS =====

Always Retry baseline: ₹3,545,565.84

===== THRESHOLD RESULTS =====
 threshold  fallback_transactions  total_net_recovery  improvement_vs_always_retry_pct
         0                      0          3454938.99                            -2.56
        10                      6          3449750.13                            -2.70
        25                     13          3459027.71                            -2.44
        50                     16          3467121.87                            -2.21
       100                     26          3456369.74                            -2.52
       250                     39          3425560.91                            -3.38
       500                     48          3446596.29                            -2.79
       750                     52          3443803.05                            -2.87
      1000                     53          3443816.05                            -2.87
      1500

In [18]:
print("===== ACTION UPLIFT ANALYSIS =====")

uplift = action_probabilities[
    [
        "transaction_id",
        "amount",
        "p_retry",
        "p_reminder",
        "p_escalation",
        "selected_action"
    ]
].copy()

# Probability uplift between actions
uplift["retry_vs_reminder"] = (
    uplift["p_retry"] - uplift["p_reminder"]
)

uplift["retry_vs_escalation"] = (
    uplift["p_retry"] - uplift["p_escalation"]
)

uplift["escalation_vs_reminder"] = (
    uplift["p_escalation"] - uplift["p_reminder"]
)

print("\n===== PREDICTED PROBABILITY UPLIFT =====")

print(
    uplift[
        [
            "retry_vs_reminder",
            "retry_vs_escalation",
            "escalation_vs_reminder"
        ]
    ]
    .describe()
    .round(4)
)

print("\n===== UPLIFT PERCENTILES =====")

print(
    uplift[
        [
            "retry_vs_reminder",
            "retry_vs_escalation",
            "escalation_vs_reminder"
        ]
    ]
    .quantile(
        [0.10, 0.25, 0.50, 0.75, 0.90]
    )
    .round(4)
)

print("\n===== CLOSEST ACTION COMPETITIONS =====")

print(
    uplift[
        [
            "transaction_id",
            "amount",
            "p_retry",
            "p_reminder",
            "p_escalation",
            "retry_vs_reminder",
            "retry_vs_escalation",
            "selected_action"
        ]
    ]
    .sort_values(
        "retry_vs_escalation",
        key=lambda x: x.abs()
    )
    .head(15)
    .round(4)
)

print("\n===== ACTION UPLIFT ANALYSIS COMPLETE =====")

===== ACTION UPLIFT ANALYSIS =====

===== PREDICTED PROBABILITY UPLIFT =====
       retry_vs_reminder  retry_vs_escalation  escalation_vs_reminder
count           789.0000             789.0000                789.0000
mean              0.1711               0.1246                  0.0465
std               0.0901               0.0839                  0.0873
min              -0.0718              -0.1262                 -0.2041
25%               0.1033               0.0723                 -0.0129
50%               0.1656               0.1279                  0.0474
75%               0.2366               0.1749                  0.1067
max               0.4303               0.4679                  0.3590

===== UPLIFT PERCENTILES =====
      retry_vs_reminder  retry_vs_escalation  escalation_vs_reminder
0.10             0.0611               0.0185                 -0.0666
0.25             0.1033               0.0723                 -0.0129
0.50             0.1656               0.1279          

In [19]:
print("===== BUILDING PAIRWISE ACTION PREFERENCE DATA =====")

# Training recovery data only
pairwise_train = recovery[
    recovery["transaction_id"].isin(train_transaction_ids)
].copy()

# Calculate actual net recovery
pairwise_train["net_recovery"] = (
    pairwise_train["amount_recovered"]
    - pairwise_train["intervention_cost"]
)

# Pivot actions to columns
pairwise_values = (
    pairwise_train
    .pivot(
        index="transaction_id",
        columns="action",
        values="net_recovery"
    )
)

print("\n===== ACTION VALUE TABLE =====")
print(pairwise_values.head())

# --------------------------------------------------
# Pair 1: Retry vs Reminder
# --------------------------------------------------

retry_reminder = pairwise_values[
    ["retry", "reminder"]
].dropna().copy()

retry_reminder["target"] = (
    retry_reminder["retry"]
    > retry_reminder["reminder"]
).astype(int)

print("\n===== RETRY VS REMINDER =====")
print(
    retry_reminder["target"]
    .value_counts()
    .rename({
        0: "reminder_better",
        1: "retry_better"
    })
)

# --------------------------------------------------
# Pair 2: Retry vs Escalation
# --------------------------------------------------

retry_escalation = pairwise_values[
    ["retry", "escalation"]
].dropna().copy()

retry_escalation["target"] = (
    retry_escalation["retry"]
    > retry_escalation["escalation"]
).astype(int)

print("\n===== RETRY VS ESCALATION =====")
print(
    retry_escalation["target"]
    .value_counts()
    .rename({
        0: "escalation_better",
        1: "retry_better"
    })
)

# --------------------------------------------------
# Pair 3: Reminder vs Escalation
# --------------------------------------------------

reminder_escalation = pairwise_values[
    ["reminder", "escalation"]
].dropna().copy()

reminder_escalation["target"] = (
    reminder_escalation["reminder"]
    > reminder_escalation["escalation"]
).astype(int)

print("\n===== REMINDER VS ESCALATION =====")
print(
    reminder_escalation["target"]
    .value_counts()
    .rename({
        0: "escalation_better",
        1: "reminder_better"
    })
)

print("\n===== PAIRWISE DATASET SIZES =====")
print("Retry vs Reminder:", len(retry_reminder))
print("Retry vs Escalation:", len(retry_escalation))
print("Reminder vs Escalation:", len(reminder_escalation))

print("\n===== PAIRWISE DATA BUILD COMPLETE =====")


===== BUILDING PAIRWISE ACTION PREFERENCE DATA =====

===== ACTION VALUE TABLE =====
action          escalation  reminder     retry
transaction_id                                
TXN1000006         3624.67   3638.67   3637.67
TXN1000009          -15.00   7140.68   7139.68
TXN1000012         7110.18     -1.00   7123.18
TXN1000016         3009.66   3023.66   3022.66
TXN1000017          -15.00     -1.00  14501.76

===== RETRY VS REMINDER =====
target
reminder_better    2171
retry_better        981
Name: count, dtype: int64

===== RETRY VS ESCALATION =====
target
retry_better         2615
escalation_better     537
Name: count, dtype: int64

===== REMINDER VS ESCALATION =====
target
reminder_better      2331
escalation_better     821
Name: count, dtype: int64

===== PAIRWISE DATASET SIZES =====
Retry vs Reminder: 3152
Retry vs Escalation: 3152
Reminder vs Escalation: 3152

===== PAIRWISE DATA BUILD COMPLETE =====


In [22]:
print("===== STEP 51: TRAIN PAIRWISE ACTION MODELS =====")

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# --------------------------------------------------
# 1. Create ONE feature row per transaction
# --------------------------------------------------

pair_features = ml_data[
    ml_data["transaction_id"].isin(train_transaction_ids)
].copy()

# ml_data contains 3 rows per transaction
# because each transaction has 3 recovery actions.
# We need only one copy of the transaction features.

pair_features = (
    pair_features
    .drop_duplicates(
        subset=["transaction_id"]
    )
    .copy()
)

# Remove action
pair_features = pair_features.drop(
    columns=["action"]
)

# Remove identifiers and leakage/post-action fields
drop_cols = [
    "transaction_id",
    "customer_id",
    "order_id",
    "calculated_success_rate",
    "rate_difference",
    "action_success",
    "amount_recovered",
    "time_to_recovery_min",
    "attempt_number_after_action",
    "failure_after_action",
    "stopping_rule_triggered"
]

pair_features = pair_features.drop(
    columns=[
        c for c in drop_cols
        if c in pair_features.columns
    ]
)

# Keep transaction IDs separately for alignment
pair_transaction_ids = (
    ml_data[
        ml_data["transaction_id"].isin(
            train_transaction_ids
        )
    ][["transaction_id"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

# Reset feature index
pair_features = pair_features.reset_index(
    drop=True
)

print("\nUnique training transactions:")
print(len(pair_features))

# --------------------------------------------------
# 2. Feature definitions
# --------------------------------------------------

numeric_features = [
    "amount",
    "attempt_number",
    "checkout_duration_sec",
    "hour_of_day",
    "day_of_week",
    "account_age_days",
    "total_transactions",
    "successful_transactions",
    "failed_transactions",
    "historical_success_rate",
    "average_transaction_value",
    "median_transaction_value",
    "total_spend",
    "days_since_last_success",
    "previous_recovery_attempts",
    "previous_recoveries"
]

categorical_features = [
    "currency",
    "payment_method",
    "status",
    "failure_reason",
    "failure_source",
    "failure_step",
    "subscription_status",
    "device_type",
    "customer_segment"
]

boolean_features = [
    "checkout_started",
    "is_subscription",
    "is_new_device"
]

# Convert boolean features to integers
for col in boolean_features:
    pair_features[col] = (
        pair_features[col]
        .astype(int)
    )

numeric_features = (
    numeric_features +
    boolean_features
)

# --------------------------------------------------
# 3. Preprocessing
# --------------------------------------------------

numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        )
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

pair_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numeric_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)

X_pair_train = pair_preprocessor.fit_transform(
    pair_features
)

print("\nRaw feature shape:")
print(pair_features.shape)

print("\nEncoded feature shape:")
print(X_pair_train.shape)

# --------------------------------------------------
# 4. Prepare pairwise targets USING transaction_id
# --------------------------------------------------

pair_target_table = (
    pairwise_values
    .reset_index()
)

pair_target_table = pair_target_table[
    pair_target_table["transaction_id"].isin(
        train_transaction_ids
    )
].copy()

pair_target_table = pair_target_table.set_index(
    "transaction_id"
)

# --------------------------------------------------
# 5. Create aligned targets
# --------------------------------------------------

transaction_order = pair_features.index

# Get transaction IDs in exactly the same
# order as pair_features
feature_transaction_ids = (
    ml_data[
        ml_data["transaction_id"].isin(
            train_transaction_ids
        )
    ][["transaction_id"]]
    .drop_duplicates()
    ["transaction_id"]
    .tolist()
)

# Safety check
assert len(feature_transaction_ids) == len(
    pair_features
)

# Targets in matching transaction order
y_retry_reminder = (
    pair_target_table
    .loc[
        feature_transaction_ids,
        "retry"
    ]
    >
    pair_target_table
    .loc[
        feature_transaction_ids,
        "reminder"
    ]
).astype(int).to_numpy()

y_retry_escalation = (
    pair_target_table
    .loc[
        feature_transaction_ids,
        "retry"
    ]
    >
    pair_target_table
    .loc[
        feature_transaction_ids,
        "escalation"
    ]
).astype(int).to_numpy()

y_reminder_escalation = (
    pair_target_table
    .loc[
        feature_transaction_ids,
        "reminder"
    ]
    >
    pair_target_table
    .loc[
        feature_transaction_ids,
        "escalation"
    ]
).astype(int).to_numpy()

pair_targets = {
    "retry_vs_reminder":
        y_retry_reminder,

    "retry_vs_escalation":
        y_retry_escalation,

    "reminder_vs_escalation":
        y_reminder_escalation
}

# --------------------------------------------------
# 6. Verify no NaN targets
# --------------------------------------------------

print("\n===== TARGET VALIDATION =====")

for pair_name, target in pair_targets.items():

    print(
        f"{pair_name}: "
        f"n={len(target)}, "
        f"NaN={int(pd.isna(target).sum())}"
    )

# --------------------------------------------------
# 7. Train pairwise models
# --------------------------------------------------

pair_models = {}
pair_metrics = {}

for pair_name, target in pair_targets.items():

    model = LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    )

    model.fit(
        X_pair_train,
        target
    )

    pair_models[pair_name] = model

    train_pred = model.predict(
        X_pair_train
    )

    train_prob = model.predict_proba(
        X_pair_train
    )[:, 1]

    metrics = {
        "accuracy": accuracy_score(
            target,
            train_pred
        ),
        "precision": precision_score(
            target,
            train_pred,
            zero_division=0
        ),
        "recall": recall_score(
            target,
            train_pred,
            zero_division=0
        ),
        "f1": f1_score(
            target,
            train_pred,
            zero_division=0
        ),
        "roc_auc": roc_auc_score(
            target,
            train_prob
        )
    }

    pair_metrics[pair_name] = metrics

# --------------------------------------------------
# 8. Display results
# --------------------------------------------------

print("\n===== TRAINING METRICS =====")

for pair_name, metrics in pair_metrics.items():

    print(f"\n{pair_name}")

    for metric_name, value in metrics.items():

        print(
            f"{metric_name}: {value:.4f}"
        )

print("\n===== STEP 51 COMPLETE =====")

===== STEP 51: TRAIN PAIRWISE ACTION MODELS =====

Unique training transactions:
3152

Raw feature shape:
(3152, 29)

Encoded feature shape:
(3152, 49)

===== TARGET VALIDATION =====
retry_vs_reminder: n=3152, NaN=0
retry_vs_escalation: n=3152, NaN=0
reminder_vs_escalation: n=3152, NaN=0


c:\Users\dhanu\Desktop\Razorpay\RevenueShield\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\dhanu\Desktop\Razorpay\RevenueShield\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-le


===== TRAINING METRICS =====

retry_vs_reminder
accuracy: 0.5155
precision: 0.3257
recall: 0.5199
f1: 0.4005
roc_auc: 0.5261

retry_vs_escalation
accuracy: 0.5114
precision: 0.8461
recall: 0.5025
f1: 0.6305
roc_auc: 0.5391

reminder_vs_escalation
accuracy: 0.5165
precision: 0.7585
recall: 0.5079
f1: 0.6084
roc_auc: 0.5446

===== STEP 51 COMPLETE =====


c:\Users\dhanu\Desktop\Razorpay\RevenueShield\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [23]:
print("===== STEP 52: ACTION SUCCESS SEGMENT ANALYSIS =====")

# --------------------------------------------------
# Training recovery data only
# --------------------------------------------------

segment_data = recovery[
    recovery["transaction_id"].isin(
        train_transaction_ids
    )
].copy()

# Add transaction/customer features
segment_data = segment_data.merge(
    transactions[
        [
            "transaction_id",
            "customer_id",
            "payment_method",
            "failure_reason",
            "failure_source",
            "failure_step",
            "amount",
            "is_subscription",
            "device_type"
        ]
    ],
    on="transaction_id",
    how="left"
)

segment_data = segment_data.merge(
    customers[
        [
            "customer_id",
            "customer_segment"
        ]
    ],
    on="customer_id",
    how="left"
)

# --------------------------------------------------
# Amount bands
# --------------------------------------------------

segment_data["amount_band"] = pd.cut(
    segment_data["amount"],
    bins=[
        -float("inf"),
        1000,
        5000,
        10000,
        50000,
        100000,
        float("inf")
    ],
    labels=[
        "<1K",
        "1K-5K",
        "5K-10K",
        "10K-50K",
        "50K-100K",
        "100K+"
    ]
)

# --------------------------------------------------
# Function to calculate action success
# --------------------------------------------------

def action_success_table(df, column):

    table = pd.crosstab(
        df[column],
        df["action"],
        values=df["action_success"],
        aggfunc="mean"
    )

    counts = pd.crosstab(
        df[column],
        df["action"]
    )

    table = (
        table[
            ["retry", "reminder", "escalation"]
        ]
        * 100
    )

    print(f"\n===== {column.upper()} =====")
    print(
        table.round(2)
    )

    print("\nSample counts:")
    print(
        counts[
            ["retry", "reminder", "escalation"]
        ]
    )

# --------------------------------------------------
# Analyze segments
# --------------------------------------------------

segment_columns = [
    "failure_reason",
    "failure_source",
    "failure_step",
    "payment_method",
    "amount_band",
    "customer_segment",
    "is_subscription",
    "device_type"
]

for column in segment_columns:

    action_success_table(
        segment_data,
        column
    )

print("\n===== STEP 52 COMPLETE =====")

===== STEP 52: ACTION SUCCESS SEGMENT ANALYSIS =====

===== FAILURE_REASON =====
action                 retry  reminder  escalation
failure_reason                                    
authentication_failed  58.86     40.51       50.95
insufficient_funds     56.67     42.05       46.98
issuer_decline         56.44     38.98       46.78
limit_exceeded         64.18     39.72       50.00
network_error          80.38     54.77       63.49
technical_error        65.98     49.18       54.51

Sample counts:
action                 retry  reminder  escalation
failure_reason                                    
authentication_failed    316       316         316
insufficient_funds       547       547         547
issuer_decline           590       590         590
limit_exceeded           282       282         282
network_error            367       367         367
technical_error          244       244         244

===== FAILURE_SOURCE =====
action          retry  reminder  escalation
failure_source 

In [25]:
print("===== STEP 53: RULE-BASED RECOVERY POLICIES =====")

# --------------------------------------------------
# 1. Prepare test transactions
# --------------------------------------------------

test_txns = transactions[
    transactions["transaction_id"].isin(
        test_transaction_ids
    )
].copy()

test_txns = test_txns[
    [
        "transaction_id",
        "amount",
        "failure_reason",
        "failure_source",
        "failure_step",
        "payment_method",
        "is_subscription",
        "device_type"
    ]
].copy()

print("\nTest transactions:")
print(len(test_txns))

# --------------------------------------------------
# 2. Prepare recovery outcomes
# --------------------------------------------------

test_recovery = recovery[
    recovery["transaction_id"].isin(
        test_transaction_ids
    )
].copy()

# --------------------------------------------------
# 3. Calculate net recovery directly
# --------------------------------------------------

test_recovery["net_recovery"] = (
    test_recovery["amount_recovered"]
    - test_recovery["intervention_cost"]
)

# --------------------------------------------------
# 4. Create one row per transaction
# --------------------------------------------------

net_pivot = (
    test_recovery
    .pivot_table(
        index="transaction_id",
        columns="action",
        values="net_recovery",
        aggfunc="first"
    )
    .reset_index()
)

# Make sure all action columns exist
for action in [
    "retry",
    "reminder",
    "escalation"
]:

    if action not in net_pivot.columns:
        net_pivot[action] = np.nan

# Explicitly rename action columns
net_pivot = net_pivot.rename(
    columns={
        "retry": "retry_net",
        "reminder": "reminder_net",
        "escalation": "escalation_net"
    }
)

# --------------------------------------------------
# 5. Merge test features + actual outcomes
# --------------------------------------------------

policy_test = test_txns.merge(
    net_pivot[
        [
            "transaction_id",
            "retry_net",
            "reminder_net",
            "escalation_net"
        ]
    ],
    on="transaction_id",
    how="left"
)

# --------------------------------------------------
# 6. Validate outcome data
# --------------------------------------------------

print("\n===== OUTCOME VALIDATION =====")

print(
    "Missing retry outcomes:",
    policy_test["retry_net"].isna().sum()
)

print(
    "Missing reminder outcomes:",
    policy_test["reminder_net"].isna().sum()
)

print(
    "Missing escalation outcomes:",
    policy_test["escalation_net"].isna().sum()
)

# --------------------------------------------------
# 7. Policy evaluation function
# --------------------------------------------------

def evaluate_policy(
    df,
    policy_name,
    action_series
):

    result = df.copy()

    result["selected_action"] = (
        action_series.values
    )

    result["selected_net_recovery"] = 0.0

    for action in [
        "retry",
        "reminder",
        "escalation"
    ]:

        mask = (
            result["selected_action"]
            == action
        )

        result.loc[
            mask,
            "selected_net_recovery"
        ] = result.loc[
            mask,
            f"{action}_net"
        ]

    total = result[
        "selected_net_recovery"
    ].sum()

    average = result[
        "selected_net_recovery"
    ].mean()

    success_rate = (
        result[
            "selected_net_recovery"
        ] > 0
    ).mean()

    print(
        f"{policy_name:<35}"
        f" ₹{total:,.2f}"
        f" | Avg: ₹{average:,.2f}"
        f" | Success: {success_rate:.2%}"
    )

    return result

# --------------------------------------------------
# 8. Policy 1 — Always Retry
# --------------------------------------------------

results = {}

results["always_retry"] = evaluate_policy(
    policy_test,
    "Always Retry",
    pd.Series(
        "retry",
        index=policy_test.index
    )
)

# --------------------------------------------------
# 9. Policy 2 — Network errors -> Retry
# --------------------------------------------------
# This is intentionally identical to Always Retry
# for now. It validates the policy framework.

results["network_retry"] = evaluate_policy(
    policy_test,
    "Network Error -> Retry",
    pd.Series(
        np.where(
            policy_test["failure_reason"]
            == "network_error",
            "retry",
            "retry"
        ),
        index=policy_test.index
    )
)

# --------------------------------------------------
# 10. Policy 3 — 100K+ -> Escalation
# --------------------------------------------------

results["100k_escalation"] = evaluate_policy(
    policy_test,
    "100K+ -> Escalation",
    pd.Series(
        np.where(
            policy_test["amount"] >= 100000,
            "escalation",
            "retry"
        ),
        index=policy_test.index
    )
)

# --------------------------------------------------
# 11. Policy 4 — Subscription -> Retry
# --------------------------------------------------

results["subscription_retry"] = evaluate_policy(
    policy_test,
    "Subscription -> Retry",
    pd.Series(
        np.where(
            policy_test["is_subscription"],
            "retry",
            "retry"
        ),
        index=policy_test.index
    )
)

print("\n===== STEP 53 COMPLETE =====")

===== STEP 53: RULE-BASED RECOVERY POLICIES =====

Test transactions:
789

===== OUTCOME VALIDATION =====
Missing retry outcomes: 0
Missing reminder outcomes: 0
Missing escalation outcomes: 0
Always Retry                        ₹3,545,565.84 | Avg: ₹4,493.75 | Success: 63.75%
Network Error -> Retry              ₹3,545,565.84 | Avg: ₹4,493.75 | Success: 63.75%
100K+ -> Escalation                 ₹3,545,552.84 | Avg: ₹4,493.73 | Success: 63.75%
Subscription -> Retry               ₹3,545,565.84 | Avg: ₹4,493.75 | Success: 63.75%

===== STEP 53 COMPLETE =====


In [26]:
print("===== STEP 55: FINALIZE AND SAVE ML MODELS =====")

import os
import joblib
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression

# ==================================================
# 1. Prepare training data
# ==================================================

final_train = ml_data[
    ml_data["transaction_id"].isin(train_transaction_ids)
].copy()

# One feature row per transaction
final_train = (
    final_train
    .drop_duplicates(
        subset=["transaction_id"]
    )
    .copy()
)

# Keep transaction IDs separately
final_transaction_ids = final_train[
    "transaction_id"
].copy()

# Remove action and identifiers/leakage
drop_cols = [
    "transaction_id",
    "customer_id",
    "order_id",
    "action",
    "calculated_success_rate",
    "rate_difference",
    "action_success",
    "amount_recovered",
    "time_to_recovery_min",
    "attempt_number_after_action",
    "failure_after_action",
    "stopping_rule_triggered"
]

X_final = final_train.drop(
    columns=[
        c for c in drop_cols
        if c in final_train.columns
    ]
)

# ==================================================
# 2. Feature definitions
# ==================================================

numeric_features = [
    "amount",
    "attempt_number",
    "checkout_duration_sec",
    "hour_of_day",
    "day_of_week",
    "account_age_days",
    "total_transactions",
    "successful_transactions",
    "failed_transactions",
    "historical_success_rate",
    "average_transaction_value",
    "median_transaction_value",
    "total_spend",
    "days_since_last_success",
    "previous_recovery_attempts",
    "previous_recoveries"
]

categorical_features = [
    "currency",
    "payment_method",
    "status",
    "failure_reason",
    "failure_source",
    "failure_step",
    "subscription_status",
    "device_type",
    "customer_segment"
]

boolean_features = [
    "checkout_started",
    "is_subscription",
    "is_new_device"
]

# Convert booleans to integers
for col in boolean_features:
    X_final[col] = X_final[col].astype(int)

numeric_features = (
    numeric_features +
    boolean_features
)

# ==================================================
# 3. Build preprocessing pipeline
# ==================================================

numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        )
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

final_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numeric_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)

X_encoded = final_preprocessor.fit_transform(
    X_final
)

print("\nTraining transactions:")
print(len(X_final))

print("\nRaw features:")
print(X_final.shape[1])

print("\nEncoded features:")
print(X_encoded.shape[1])

# ==================================================
# 4. Prepare action targets
# ==================================================

final_values = (
    pairwise_values
    .loc[
        final_transaction_ids,
        [
            "retry",
            "reminder",
            "escalation"
        ]
    ]
)

# ==================================================
# 5. Train final action-success models
# ==================================================

final_models = {}

action_targets = {
    "retry": (
        recovery[
            recovery["transaction_id"].isin(
                train_transaction_ids
            )
            & (recovery["action"] == "retry")
        ]
        .set_index("transaction_id")
        .loc[
            final_transaction_ids,
            "action_success"
        ]
        .astype(int)
        .to_numpy()
    ),

    "reminder": (
        recovery[
            recovery["transaction_id"].isin(
                train_transaction_ids
            )
            & (recovery["action"] == "reminder")
        ]
        .set_index("transaction_id")
        .loc[
            final_transaction_ids,
            "action_success"
        ]
        .astype(int)
        .to_numpy()
    ),

    "escalation": (
        recovery[
            recovery["transaction_id"].isin(
                train_transaction_ids
            )
            & (recovery["action"] == "escalation")
        ]
        .set_index("transaction_id")
        .loc[
            final_transaction_ids,
            "action_success"
        ]
        .astype(int)
        .to_numpy()
    )
}

for action, target in action_targets.items():

    model = LogisticRegression(
        max_iter=2000,
        random_state=42
    )

    model.fit(
        X_encoded,
        target
    )

    final_models[action] = model

    print(
        f"\n{action.capitalize()} model trained"
    )

# ==================================================
# 6. Create models directory
# ==================================================

models_dir = os.path.join(
    "..",
    "models"
)

os.makedirs(
    models_dir,
    exist_ok=True
)

# ==================================================
# 7. Save preprocessing + models
# ==================================================

joblib.dump(
    final_preprocessor,
    os.path.join(
        models_dir,
        "recovery_preprocessor.joblib"
    )
)

for action, model in final_models.items():

    joblib.dump(
        model,
        os.path.join(
            models_dir,
            f"{action}_model.joblib"
        )
    )

# ==================================================
# 8. Save metadata
# ==================================================

metadata = {
    "model_type": "LogisticRegression",
    "training_transactions": len(final_train),
    "encoded_features": X_encoded.shape[1],
    "actions": [
        "retry",
        "reminder",
        "escalation"
    ],
    "random_state": 42,
    "excluded_leakage_fields": [
        "amount_recovered",
        "time_to_recovery_min",
        "attempt_number_after_action",
        "failure_after_action",
        "stopping_rule_triggered"
    ]
}

joblib.dump(
    metadata,
    os.path.join(
        models_dir,
        "model_metadata.joblib"
    )
)

# ==================================================
# 9. Verify files
# ==================================================

print("\n===== SAVED MODEL FILES =====")

for filename in sorted(
    os.listdir(models_dir)
):

    print(
        filename
    )

print("\n===== STEP 55 COMPLETE =====")

===== STEP 55: FINALIZE AND SAVE ML MODELS =====

Training transactions:
3152

Raw features:
29

Encoded features:
49


c:\Users\dhanu\Desktop\Razorpay\RevenueShield\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



Retry model trained


c:\Users\dhanu\Desktop\Razorpay\RevenueShield\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



Reminder model trained

Escalation model trained

===== SAVED MODEL FILES =====
escalation_model.joblib
model_metadata.joblib
recovery_preprocessor.joblib
reminder_model.joblib
retry_model.joblib

===== STEP 55 COMPLETE =====


c:\Users\dhanu\Desktop\Razorpay\RevenueShield\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
